# RQ1 robustness tests

Checks reported in Section 4.4: SOC3 fixed effects, individual-occupation
exclusions, GPT-4 Beta, core-weighted shares, and median wage.

Set `BASE_DIR` to the project folder containing the three frozen input CSVs,
then run the notebook from top to bottom. The notebook displays results and
retains unrounded result tables in memory; it does not export files.

M0 contains aggregate exposure; M1 adds mechanism composition; M2 adds
structural states and conditional importance positions. All fits use OLS with
HC3 covariance, normal-based coefficient intervals, and Wald F joint tests.
The input outcomes are annualized log changes. Coefficients, standard errors,
and interval bounds are displayed on the manuscript's scale (×100).

## 1. Inputs and common samples

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

BASE_DIR = Path(
    "/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-full"
)

configuration_data = pd.read_csv(
    BASE_DIR / "05_Layer3_Task_Configuration/Output/Occupational_Task_Affordance_Configuration_v6.csv",
    dtype={"soc6": "string"},
)

exposure_data = pd.read_csv(
    BASE_DIR / "07_GenAI_Exposure/Output/Occupational_Configuration_Exposure_v1.csv",
    dtype={"soc6": "string"},
)

outcome_data = pd.read_csv(
    BASE_DIR / "08_Labor_Outcomes/Output/OEWS_Occupational_Outcomes_v1.csv",
    dtype={"soc6": "string"},
)

### Model data and samples

In [7]:
mechanisms = ["automation", "augmentation", "limited"]
outcomes = {
    "Employment": "employment_ann_log_change",
    "Wage": "mean_wage_ann_log_change",
    "Median wage": "median_wage_ann_log_change",
}

model_data = (
    configuration_data
    .drop(columns=["human_beta", "gpt4_beta"], errors="ignore")
    .merge(
        exposure_data[["soc6", "human_beta", "gpt4_beta"]],
        on="soc6", how="left",
    )
    .merge(
        outcome_data[["soc6", *outcomes.values()]],
        on="soc6", how="left",
    )
)

eligible = model_data[
    ["human_beta", *[f"{m}_share" for m in mechanisms]]
].notna().all(axis=1)

for mechanism in mechanisms:
    present = model_data[f"{mechanism}_count"].gt(0)
    position = model_data[f"{mechanism}_position"]
    eligible &= ~(present & position.isna())
    model_data[f"{mechanism}_position_centered"] = (
        position - 0.5
    ).where(present, 0.0)

model_data["automation_present"] = model_data["automation_count"].gt(0).astype(int)
model_data["augmentation_absent"] = model_data["augmentation_count"].eq(0).astype(int)
model_data["limited_absent"] = model_data["limited_count"].eq(0).astype(int)
model_data["soc3"] = model_data["soc6"].str[:4].astype(object)

samples = {
    outcome: model_data.loc[eligible & model_data[column].notna()].copy()
    for outcome, column in outcomes.items()
}

### Shared model function and reference models

In [8]:
main_outcomes = ["Employment", "Wage"]
composition_terms = ["automation_share", "augmentation_share"]
state_terms = ["automation_present", "augmentation_absent", "limited_absent"]
position_terms = [f"{m}_position_centered" for m in mechanisms]

def fit_models(frame, outcome, exposure="human_beta",
               shares=composition_terms, soc3=False):
    terms = {
        "M0": [exposure],
        "M1": [exposure, *shares],
        "M2": [exposure, *shares, *state_terms, *position_terms],
    }
    fitted = {}

    for name, predictors in terms.items():
        if soc3:
            formula = (
                f"{outcomes[outcome]} ~ {' + '.join(predictors)} + C(soc3)"
            )
            model = smf.ols(formula, data=frame, missing="raise")
        else:
            model = sm.OLS(
                frame[outcomes[outcome]],
                sm.add_constant(frame[predictors], has_constant="add"),
                missing="raise",
            )

        fitted[name] = model.fit(cov_type="HC3", use_t=False)

    return fitted

primary_models = {
    outcome: fit_models(frame, outcome)
    for outcome, frame in samples.items()
}

### Shared joint tests

In [9]:
def joint_tests(fitted_models, outcome, specification, shares=composition_terms):
    blocks = [
        ("mechanism composition", "M1", "M1 vs. M0", shares),
        ("structural states and importance positions", "M2", "M2 vs. M1",
         state_terms + position_terms),
        ("task-affordance configuration", "M2", "M2 vs. M0",
         shares + state_terms + position_terms),
        ("importance positions", "M2", "within M2: joint test", position_terms),
    ]
    rows = []

    for block, name, comparison, terms in blocks:
        model = fitted_models[name]
        test = model.wald_test(
            ", ".join(f"{term} = 0" for term in terms),
            use_f=True, scalar=True,
        )
        rows.append({
            "Outcome": outcome,
            "Specification": specification,
            "Block": block,
            "Comparison": comparison,
            "Model": name,
            "N": int(model.nobs),
            "F": float(test.statistic),
            "df_num": int(test.df_num),
            "df_denom": int(test.df_denom),
            "p": float(test.pvalue),
        })

    return pd.DataFrame(rows)

primary_tests = pd.concat([
    joint_tests(fitted, outcome, "Primary")
    for outcome, fitted in primary_models.items()
], ignore_index=True)

### SOC3 fixed effects

In [10]:
soc3_models = {}
singleton_rows = []
soc3_tests = [
    primary_tests.loc[primary_tests["Outcome"].isin(main_outcomes)]
]

for outcome in main_outcomes:
    frame = samples[outcome]
    keep = frame.groupby("soc3")["soc3"].transform("size").ge(2)
    matched = frame.loc[keep]

    singleton_rows.append(
        frame.loc[~keep, ["soc6", "soc3"]].assign(Outcome=outcome)
    )

    soc3_models[outcome] = {
        "Primary": primary_models[outcome],
        "Matched sample": fit_models(matched, outcome),
        "SOC3 FE": fit_models(matched, outcome, soc3=True),
    }

    for specification in ["Matched sample", "SOC3 FE"]:
        soc3_tests.append(
            joint_tests(
                soc3_models[outcome][specification],
                outcome,
                specification,
            )
        )

soc3_tests = pd.concat(soc3_tests, ignore_index=True)
singletons = pd.concat(singleton_rows, ignore_index=True)

### SOC3 coefficient comparison

In [12]:
benchmark_models = {
    outcome: fit_models(samples[outcome], outcome, exposure="gpt4_beta")
    for outcome in main_outcomes
}
benchmark_tests = pd.concat([
    joint_tests(fitted, outcome, "GPT-4 Beta")
    for outcome, fitted in benchmark_models.items()
], ignore_index=True)

table4_results = pd.concat([soc3_tests, benchmark_tests], ignore_index=True)

comparison_columns = {
    "Primary": ("Exposure comparison", "Human Beta"),
    "GPT-4 Beta": ("Exposure comparison", "GPT-4 Beta"),
    "Matched sample": ("SOC3 comparison", "Without SOC3 FE"),
    "SOC3 FE": ("SOC3 comparison", "With SOC3 FE"),
}


def format_wald(r):
    p = "< .001" if r.p < .001 else f"= {r.p:.4f}".replace("0.", ".")
    return f"F ({int(r.df_num)}, {int(r.df_denom)}) = {r.F:.3f};\np {p}"


table4_rows = []

for outcome in main_outcomes:
    part = table4_results.loc[table4_results["Outcome"].eq(outcome)]
    counts = part.groupby("Specification")["N"].first()

    table4_rows.append({
        ("Block tested", ""): outcome,
        ("Comparison", ""): "N",
        **{column: int(counts[spec])
           for spec, column in comparison_columns.items()},
    })

    for (block, comparison), tests in part.groupby(
        ["Block", "Comparison"], sort=False
    ):
        tests = tests.set_index("Specification")
        table4_rows.append({
            ("Block tested", ""): block,
            ("Comparison", ""): comparison,
            **{column: format_wald(tests.loc[spec])
               for spec, column in comparison_columns.items()},
        })

table_4 = pd.DataFrame(table4_rows)
table_4.columns = pd.MultiIndex.from_tuples(table_4.columns)

display(
    table_4.style.hide(axis="index")
    .set_caption("Table 4. Joint tests by exposure measure and SOC3 FE.")
    .set_properties(**{
        "font-size": "12px", "text-align": "left", "white-space": "pre-wrap"
    })
)

print(
    "Note. Entries are HC3-robust Wald F tests. Both specifications in the "
    "SOC3 comparison use Human Beta and the same occupations within each outcome."
)

Note. Entries are HC3-robust Wald F tests. Both specifications in the SOC3 comparison use Human Beta and the same occupations within each outcome.


In [13]:
influential_socs = {}
omission_rows = []

for outcome in main_outcomes:
    frame = samples[outcome]
    selected = set()

    for name in ["M1", "M2"]:
        influence = primary_models[outcome][name].get_influence()
        selected.update(
            pd.Series(np.asarray(influence.cooks_distance[0])).nlargest(5).index
        )
        selected.update(
            pd.Series(np.asarray(influence.hat_matrix_diag)).nlargest(3).index
        )
        selected.update(
            np.abs(np.asarray(influence.dfbetas)[:, 2:]).argmax(axis=0)
        )

    influential_socs[outcome] = sorted(frame.iloc[list(selected)]["soc6"])

    for soc6 in influential_socs[outcome]:
        reduced = frame.loc[frame["soc6"].ne(soc6)]
        omission_rows.append(
            joint_tests(fit_models(reduced, outcome), outcome, soc6)
        )

omission_tests = pd.concat(omission_rows, ignore_index=True).rename(
    columns={"Specification": "Excluded SOC6"}
)

keys = ["Outcome", "Block", "Comparison"]
ranges = omission_tests.groupby(keys, sort=False)["p"].agg(
    **{"Minimum p": "min", "Maximum p": "max"}
).reset_index()

table5_results = (
    primary_tests.loc[
        primary_tests["Outcome"].isin(main_outcomes), keys + ["p"]
    ]
    .rename(columns={"p": "Full-sample p"})
    .merge(ranges, on=keys, how="left", sort=False)
)

def format_p(p):
    return "< .001" if p < .001 else f"{p:.4f}".lstrip("0")

table5_rows = []

for (block, comparison), part in table5_results.groupby(
    ["Block", "Comparison"], sort=False
):
    part = part.set_index("Outcome")
    row = {("Block tested", ""): block, ("Comparison", ""): comparison}

    for outcome in main_outcomes:
        r = part.loc[outcome]
        low, high = r["Minimum p"], r["Maximum p"]
        row[(outcome, "Full-sample p")] = format_p(r["Full-sample p"])
        row[(outcome, "Exclusion p range")] = (
            "All < .001" if high < .001
            else f"{format_p(low)} – {format_p(high)}"
        )

    table5_rows.append(row)

table_5 = pd.DataFrame(table5_rows)
table_5.columns = pd.MultiIndex.from_tuples(table_5.columns)

display(
    table_5.style.hide(axis="index")
    .set_caption("Table 5. Joint tests after individual occupation exclusion.")
    .set_properties(**{"font-size": "12px", "text-align": "left"})
)

print(
    "Note. For each outcome, occupations were selected using the five largest "
    "Cook’s distances, the three highest leverage values, and the largest absolute "
    "DFBETAS for each configuration coefficient, separately in M1 and M2. "
    f"Combining these selections identified {len(influential_socs['Employment'])} "
    f"distinct occupations for employment and {len(influential_socs['Wage'])} for wage. "
    "Each was excluded one at a time from the corresponding full sample "
    f"(employment: N = {len(samples['Employment'])}; "
    f"wage: N = {len(samples['Wage'])})."
)

Note. For each outcome, occupations were selected using the five largest Cook’s distances, the three highest leverage values, and the largest absolute DFBETAS for each configuration coefficient, separately in M1 and M2. Combining these selections identified 17 distinct occupations for employment and 13 for wage. Each was excluded one at a time from the corresponding full sample (employment: N = 699; wage: N = 697).


### Core-weighted-share check

In [14]:
weighted_shares = [
    term.replace("_share", "_core_weighted_share")
    for term in composition_terms
]

weighted_models = {}
weighted_tests = [
    primary_tests.loc[
        primary_tests["Outcome"].isin(main_outcomes)
    ].assign(Specification="Unweighted shares")
]

for outcome in main_outcomes:
    weighted_models[outcome] = fit_models(
        samples[outcome], outcome, shares=weighted_shares
    )
    weighted_tests.append(
        joint_tests(
            weighted_models[outcome], outcome,
            "Core-weighted shares", shares=weighted_shares,
        )
    )

weighted_tests = pd.concat(weighted_tests, ignore_index=True)

for outcome, part in weighted_tests.groupby("Outcome", sort=False):
    order = pd.MultiIndex.from_frame(
        part[["Block", "Comparison"]].drop_duplicates()
    )
    table = (
        part.assign(Result=part.apply(format_wald, axis=1))
        .pivot(index=["Block", "Comparison"],
               columns="Specification", values="Result")
        .reindex(index=order, columns=["Unweighted shares", "Core-weighted shares"])
        .rename_axis(columns=None)
        .reset_index()
        .rename(columns={"Block": "Block tested"})
    )

    display(
        table.style.hide(axis="index")
        .set_caption(f"{outcome}: core-weighted-share check (N = {len(samples[outcome])})")
        .set_properties(**{
            "font-size": "12px", "text-align": "left", "white-space": "pre-wrap"
        })
    )

Block tested,Comparison,Unweighted shares,Core-weighted shares
mechanism composition,M1 vs. M0,"F (2, 695) = 21.107; p < .001","F (2, 695) = 20.292; p < .001"
structural states and importance positions,M2 vs. M1,"F (6, 689) = 4.446; p < .001","F (6, 689) = 4.888; p < .001"
task-affordance configuration,M2 vs. M0,"F (8, 689) = 9.418; p < .001","F (8, 689) = 9.395; p < .001"
importance positions,within M2: joint test,"F (3, 689) = 6.010; p < .001","F (3, 689) = 6.953; p < .001"


Block tested,Comparison,Unweighted shares,Core-weighted shares
mechanism composition,M1 vs. M0,"F (2, 693) = 4.954; p = .0073","F (2, 693) = 5.723; p = .0034"
structural states and importance positions,M2 vs. M1,"F (6, 687) = 1.754; p = .1060","F (6, 687) = 1.707; p = .1166"
task-affordance configuration,M2 vs. M0,"F (8, 687) = 2.425; p = .0137","F (8, 687) = 2.581; p = .0088"
importance positions,within M2: joint test,"F (3, 687) = 1.252; p = .2898","F (3, 687) = 1.049; p = .3700"
